In [ ]:
import pandas as pd
import os
import glob
from read_inf_list import get_inference_file_info
from dataloader_inf import open_dataset_inf, open_datasets_inf
from validation_statistics import create_stat_csv_plot_multiple_inf
from distribution_comparison import create_distribution_plots

In [ ]:
import pandas as pd
import os
import glob
import ast  # To safely evaluate string representations of lists

def get_inference_file_info(infile):
    # Read the CSV file into a DataFrame
    df = pd.read_csv(infile, comment='#')
    # Initialize lists to collect data
    inference_files_paths = []
    inference_run_id      = []
    inferece_epoch        = []
    plot_titles           = []
    # Variables to check for consistency
    inf_date_set   = set()
    inf_length_set = set()
    variables_set  = set()
    for i in df.index:
        inf_path  = df['inference_file'][i]
        # Safely parse the string representation of a path if needed
        if isinstance(inf_path, str):
            try:
                # Handle cases where the path might look like a string with extra quotes
                inf_path = ast.literal_eval(inf_path)
            except (ValueError, SyntaxError):
                pass  # If parsing fails, assume it's already a clean string
        inf_file  = inf_path.split('/')[-1]
        inf_vars  = df['variables'][i]
        plt_title = df['plot_title'][i]
        # Handle default values for plot_title
        if plt_title == '' or plt_title == '*':
            plt_title = inf_file[-21:-4]
        # Handle default values or parse provided variables
        #print(inf_vars)
        if inf_vars == '' or inf_vars == '*':
            inf_vars = ['salinity', 
                        'temperature', 
                        'u_eastward', 
                        'v_northward']
        else:
            # Safely parse the string representation of a list into an actual list
            if isinstance(inf_vars, str):
                #print(f'isinstance(inf_vars, str) == {isinstance(inf_vars, str)}')
                #print(inf_vars)
                try:
                    #inf_vars = ast.literal_eval(inf_vars)  # Convert string to list
                    # Ensure the string looks like a list before parsing
                    if not (inf_vars.startswith('[') and inf_vars.endswith(']')):
                        inf_vars = [inf_vars]  # Convert single variable to a list
                    else:
                        # Fix missing commas between items
                        inf_vars = inf_vars.replace("''", "', '")
                    # Convert string to a list
                    inf_vars = ast.literal_eval(inf_vars)
                except (ValueError, SyntaxError):
                    raise ValueError(f"Invalid format for variables: {inf_vars}")
            # Ensure variables are sorted for consistency
            inf_vars = sorted([var.strip() for var in inf_vars])
        # Extract information from the file name
        inf_date   = inf_file[0:10]
        inf_length = inf_file[11:14]
        inf_run_id = inf_file[15:20]
        inf_epoch  = inf_file[21:33]
        # Collect the extracted information
        inference_files_paths.append(inf_path)
        inference_run_id.append(inf_run_id)
        inferece_epoch.append(inf_epoch)
        plot_titles.append(plt_title)
        # Add to consistency check sets
        inf_date_set.add(inf_date)
        inf_length_set.add(inf_length)
        variables_set.add(tuple(inf_vars))  # Add variables as a tuple for immutability
    # Check if all inference files have the same date, length, and variables
    if len(inf_date_set) > 1:
        raise ValueError(f"Inconsistent inference dates found: {inf_date_set}")
    if len(inf_length_set) > 1:
        raise ValueError(f"Inconsistent inference lengths found: {inf_length_set}")
    if len(variables_set) > 1:
        raise ValueError(f"Inconsistent variables found: {variables_set}")
    # Prepare the plot dictionary
    plot_dict = {
        'inf_date'            : inf_date_set.pop(),         # Get the single consistent date
        'variables'           : list(variables_set.pop()),  # Convert the single consistent tuple back to a list
        'inf_length'          : inf_length_set.pop(),       # Get the single consistent length
        'number_of_inf_files' : len(inference_files_paths)
    }
    # Return the requested data
    return inference_files_paths, plot_titles, inference_run_id, inferece_epoch, plot_dict

def create_directory(path: str):
        """
        Create a directory if it does not exist, and handle the case where it already exists.
        """
        try:
            # Try to create the directory
            os.makedirs(path, exist_ok=False)
            print(f"Directory '{path}' created successfully.")
        except FileExistsError:
            # Handle the case where the directory already exists
            print(f"Directory '{path}' already exists. Skipping creation.")



In [ ]:
inference_files_paths, plot_titles, inference_run_id, inferece_epoch, plot_dict = get_inference_file_info(infile='inference_list.csv')

In [ ]:
plot_titles, plot_dict

In [ ]:
import xarray as xr
inf_2m = xr.open_dataset('/lustre/storeB/project/fou/hi/foccus/experiments/2m-depth/results/2024-04-02_48h_e655e_e013_s054000.nc')
#inf_2m

In [ ]:
infile = 'inference_list.csv'
#df = pd.read_csv(infile, comment='#')
inference_files_paths, plot_titles, inference_run_id, inferece_epoch, plot_dict = get_inference_file_info(infile)
norkyst3, havbris_combined = open_datasets_inf(inference_files_paths,
                    truth_norkyst3_path = '/lustre/storeB/project/fou/hi/foccus/datasets/symlinks/norkystv3-hindcast/',
                    truth_file_name_template = "{year}/norkyst800-{year}{month:02d}{day:02d}.nc",
                    variables=plot_dict['variables'],
                    s_rho = -1,
                    crop_border = True,
                    debug=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

variable = 'temperature'

'''plt.imshow(norkyst3[variable][0,:,:], vmin=0, vmax=10)
plt.colorbar()
plt.xlim(341,864)
plt.ylim(0,428)
plt.show()'''

'''plt.imshow(np.flipud(havbris_combined[variable][0,0,:,:]), vmin=0, vmax=10)
plt.colorbar()
plt.show()'''

Y_min = 50
Y_max = 305
X_min = 450
X_max = 755

'''plt.imshow(norkyst3[variable][0,:,:], vmin=0, vmax=10)
plt.colorbar()
plt.xlim(X_min,X_max) # Should be total of 305
plt.ylim(Y_min,Y_max)  # Should be total of 255
plt.show()'''

subset = norkyst3.isel(Y=slice(Y_min, Y_max), X=slice(X_min, X_max))
plt.figure(figsize=(10, 8))
plt.title(f"Difference between norkyst3 and havbris_combined for {variable}")
plt.imshow(np.flipud(subset[variable][1,:,:]) - np.flipud(havbris_combined[variable][0,1,:,:]), vmin=-1, vmax=1, cmap='bwr')
plt.colorbar()
plt.show()

variable = 'salinity'

Y_min = 50
Y_max = 305
X_min = 450
X_max = 755

subset = norkyst3.isel(Y=slice(Y_min, Y_max), X=slice(X_min, X_max))
plt.figure(figsize=(10, 8))
plt.title(f"Difference between norkyst3 and havbris_combined for {variable}")
plt.imshow(np.flipud(subset[variable][1,:,:]) - np.flipud(havbris_combined[variable][0,1,:,:]), vmin=-5, vmax=5, cmap='bwr')
plt.colorbar()
plt.show()

In [ ]:
norkyst3

In [ ]:
havbris_combined

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(norkyst3.temperature[0,:,:])

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(havbris_combined.temperature[0,0,:,:])

In [ ]:
import numpy as np

def subset_dataset_based_on_dataset(dataset_to_be_subset, dataset_reference):
    # TODO: Fix lat and lon name as input-variables?
    lon_min, lon_max = np.min(dataset_reference['lon']), np.max(dataset_reference['lon'])
    lat_min, lat_max = np.min(dataset_reference['lat']), np.max(dataset_reference['lat'])
    # Select only the relevant portion of the dataset
    subset = dataset_to_be_subset.where(
        (dataset_to_be_subset['lon'] >= lon_min) & (dataset_to_be_subset['lon'] <= lon_max) &
        (dataset_to_be_subset['lat'] >= lat_min) & (dataset_to_be_subset['lat'] <= lat_max),
        drop=True)
    return subset

In [ ]:
norkyst3_sub = subset_dataset_based_on_dataset(norkyst3, havbris_combined)

In [ ]:
import numpy as np

def subset_dataset_based_on_indices(dataset_to_be_subset, dataset_reference):
    """
    Subset a dataset based on the max/min lat/lon bounds of a reference dataset, but crop using X and Y indices.

    Parameters:
        dataset_to_be_subset (xarray.Dataset): Dataset to be subset.
        dataset_reference (xarray.Dataset): Reference dataset defining the smaller domain.

    Returns:
        xarray.Dataset: Subset of the original dataset cropped by X and Y indices.
    """
    # Extract lat/lon bounds of the reference dataset
    lon_ref = dataset_reference['lon'].values
    lat_ref = dataset_reference['lat'].values
    lon_min, lon_max = lon_ref.min(), lon_ref.max()
    lat_min, lat_max = lat_ref.min(), lat_ref.max()

    # Extract lat/lon values from the target dataset
    lon_sub = dataset_to_be_subset['lon'].values
    lat_sub = dataset_to_be_subset['lat'].values

    # Find the corresponding X and Y indices in the target dataset
    mask = (
        (lon_sub >= lon_min) & (lon_sub <= lon_max) &
        (lat_sub >= lat_min) & (lat_sub <= lat_max)
    )
    Y_indices, X_indices = np.where(mask)

    # Check if no points match the mask
    if len(Y_indices) == 0 or len(X_indices) == 0:
        raise ValueError("No matching data found. Ensure the datasets overlap.")

    # Find the min and max X and Y indices
    Y_min, Y_max = Y_indices.min(), Y_indices.max()
    X_min, X_max = X_indices.min(), X_indices.max()

    # Crop the dataset using the X and Y indices
    subset = dataset_to_be_subset.isel(Y=slice(Y_min, Y_max), X=slice(X_min, X_max))

    # Debug print statements
    print("Reference domain bounds (lat/lon):")
    print(f"  lon_min: {lon_min}, lon_max: {lon_max}")
    print(f"  lat_min: {lat_min}, lat_max: {lat_max}")
    print("Subset domain bounds (X/Y indices):")
    print(f"  X_min: {X_min}, X_max: {X_max}")
    print(f"  Y_min: {Y_min}, Y_max: {Y_max}")

    return subset, mask

In [ ]:
# Extract lon and lat from the reference dataset
lon_ref = norkyst3_sub['lon']
lat_ref = norkyst3_sub['lat']

# Find the min and max lon/lat values in the reference dataset
lon_min, lon_max = lon_ref.min().item(), lon_ref.max().item()
lat_min, lat_max = lat_ref.min().item(), lat_ref.max().item()

print(f"Reference domain: lon [{lon_min}, {lon_max}], lat [{lat_min}, {lat_max}]")

In [ ]:
norkyst3_sub, mask = subset_dataset_based_on_indices(norkyst3, havbris_combined)

In [ ]:
havbris_combined

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.imshow(norkyst3.temperature[0,:,:], vmin=0, vmax=10)
plt.colorbar()
plt.xlim(341,864)
plt.ylim(0,428)
plt.show()

plt.imshow(np.flipud(havbris_combined.temperature[0,0,:,:]), vmin=0, vmax=10)
plt.colorbar()
plt.show()

Y_min = 50
Y_max = 305
X_min = 450
X_max = 755

plt.imshow(norkyst3.temperature[0,:,:], vmin=0, vmax=10)
plt.colorbar()
plt.xlim(X_min,X_max) # Should be total of 305
plt.ylim(Y_min,Y_max)  # Should be total of 255
plt.show()

subset = norkyst3.isel(Y=slice(Y_min, Y_max), X=slice(X_min, X_max))
plt.imshow(np.flipud(subset.lat) - np.flipud(havbris_combined.lat), vmin=-0.001, vmax=0.001, cmap='bwr')
plt.colorbar()
plt.show()
plt.imshow(np.flipud(subset.lon) - np.flipud(havbris_combined.lon), vmin=-0.001, vmax=0.001, cmap='bwr')
plt.colorbar()
plt.show()
plt.imshow(np.flipud(subset.lat))
plt.colorbar()
plt.show()
plt.imshow(np.flipud(havbris_combined.lat))
plt.colorbar()
plt.show()

subset = norkyst3.isel(Y=slice(Y_min, Y_max), X=slice(X_min, X_max))
plt.imshow(np.flipud(subset.temperature[0,:,:]) - np.flipud(havbris_combined.temperature[0,0,:,:]), vmin=-1, vmax=1, cmap='bwr')
plt.colorbar()
plt.show()

In [ ]:
plt.imshow(mask)

In [ ]:
havbris_combined

In [ ]:
norkyst3_sub

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(np.flipud(havbris_combined.temperature[0,0,:,:]))

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(norkyst3_sub.temperature[0,:,:])
plt.xlim(105,410)
plt.ylim(50,300)

In [ ]:
def create_validation_stats_from_infile():
    infile = 'inference_list.csv'
    #df = pd.read_csv(infile, comment='#')
    inference_files_paths, plot_titles, inference_run_id, inferece_epoch, plot_dict = get_inference_file_info(infile)
    norkyst3, havbris_combined = open_datasets_inf(inference_files_paths,
                        truth_norkyst3_path = '/lustre/storeB/project/fou/hi/foccus/datasets/symlinks/norkystv3-hindcast/',
                        truth_file_name_template = "{year}/norkyst800-{year}{month:02d}{day:02d}.nc",
                        variables=plot_dict['variables'],
                        s_rho = -1,
                        crop_border = True,
                        debug=True)
    
    create_stat_csv_plot_multiple_inf(norkyst3, havbris_combined, plot_dict, plot_titles)
create_validation_stats_from_infile()

In [ ]:
Y_min = 50
Y_max = 305
X_min = 450
X_max = 755

plt.imshow(norkyst3.temperature[0,:,:], vmin=0, vmax=10)
plt.colorbar()
plt.xlim(X_min,X_max) # Should be total of 305
plt.ylim(Y_min,Y_max)  # Should be total of 255
plt.show()

subset = norkyst3.isel(Y=slice(Y_min, Y_max), X=slice(X_min, X_max))

In [ ]:
Y_min = 50
Y_max = 305
X_min = 450
X_max = 755
def create_validation_stats_from_infile():
    infile = 'inference_list.csv'
    #df = pd.read_csv(infile, comment='#')
    inference_files_paths, plot_titles, inference_run_id, inferece_epoch, plot_dict = get_inference_file_info(infile)
    norkyst3, havbris_combined = open_datasets_inf(inference_files_paths,
                        truth_norkyst3_path = '/lustre/storeB/project/fou/hi/foccus/datasets/symlinks/norkystv3-hindcast/',
                        truth_file_name_template = "{year}/norkyst800-{year}{month:02d}{day:02d}.nc",
                        variables=plot_dict['variables'],
                        s_rho = -1,
                        crop_border = True,
                        debug=True)
    subset_nk3 = norkyst3.isel(Y=slice(Y_min, Y_max), X=slice(X_min, X_max))
    create_stat_csv_plot_multiple_inf(subset_nk3, havbris_combined, plot_dict, plot_titles)
create_validation_stats_from_infile()

# Need to fix to oslo-fjord! 

In [ ]:
def create_distribution_gifs(inference_path):
    havbris, norkyst3 = open_dataset_inf(inference_path,crop_border=True,
                                         variables=plot_dict['variables'],)
    create_distribution_plots(norkyst3, havbris, variables=plot_dict['variables'])

inference_path = '/lustre/storeB/project/fou/hi/foccus/experiments/oslofjorden/results/2024-04-02_72h_d24ef_e058_s220000.nc'#'/lustre/storeB/project/fou/hi/foccus/ingvild/test_infrence/results/2024-04-02_72h_e5281_e025_s098000.nc'#'/lustre/storeB/project/fou/hi/foccus/ingvild/test_infrence/results/2024-04-02_72h_18d28_e011_s049990.nc'
create_distribution_gifs(inference_path)